# Dataset Exploration — McAuley-Lab/Amazon-Reviews-2023 (raw_meta_Electronics)

Key questions:
1. What fields are available and how complete are they?
2. Is `features` / `description` rich enough for semantic retrieval?
3. What does the price / rating distribution look like?
4. What would a final embedded document look like?

## 1. Authenticate with HuggingFace

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
login(os.environ['HF_TOKEN'])
print('Authenticated')

## 2. Stream a sample — avoid downloading all 1.6M rows

In [ ]:
from datasets import load_dataset

SAMPLE_SIZE = 5_000  # bump to 50k+ once happy

ds = load_dataset(
    'McAuley-Lab/Amazon-Reviews-2023',
    'raw_meta_Electronics',
    split='full',
    streaming=True,
    trust_remote_code=True,
)

records = [row for _, row in zip(range(SAMPLE_SIZE), ds)]
print(f'Loaded {len(records):,} records')

## 3. Schema — column names and a raw record peek

In [ ]:
import pandas as pd

df = pd.DataFrame(records)
print(f'Shape: {df.shape}\n')
print('Columns and dtypes:')
print(df.dtypes.to_string())

In [ ]:
# One raw record — inspect every field
for k, v in records[10].items():
    print(f'{k:20s}: {repr(v)[:140]}')

## 4. Completeness — fill rate per field

In [ ]:
def is_empty(val):
    if val is None:
        return True
    if isinstance(val, (list, dict)) and len(val) == 0:
        return True
    if isinstance(val, str) and val.strip() == '':
        return True
    return False

completeness = {
    col: 1 - sum(is_empty(row[col]) for row in records) / len(records)
    for col in records[0].keys()
}
comp_df = pd.Series(completeness).sort_values().rename('fill_rate')
comp_df

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
comp_df.plot(kind='barh', ax=ax, color='steelblue')
ax.axvline(0.5, color='red', linestyle='--', linewidth=1, label='50% threshold')
ax.set_xlabel('Fill rate')
ax.set_title(f'Field completeness  (n={SAMPLE_SIZE:,})')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Text richness — `features` vs `description`

In [ ]:
def features_text(row):
    feats = row.get('features') or []
    return ' '.join(feats) if feats else ''

def description_text(row):
    desc = row.get('description') or []
    return ' '.join(desc) if isinstance(desc, list) else str(desc or '')

df['features_len']    = [len(features_text(r)) for r in records]
df['description_len'] = [len(description_text(r)) for r in records]
df['has_features']    = df['features_len'] > 0
df['has_description'] = df['description_len'] > 0

print(f"Has features:     {df['has_features'].mean():.1%}")
print(f"Has description:  {df['has_description'].mean():.1%}")
print(f"Has either:       {(df['has_features'] | df['has_description']).mean():.1%}")
print()
print('features_len stats:')
print(df['features_len'].describe().to_string())
print()
print('description_len stats:')
print(df['description_len'].describe().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df[df['features_len'] > 0]['features_len'].clip(upper=2000).hist(
    bins=50, ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('Features text length (non-empty, clipped 2k chars)')
axes[0].set_xlabel('characters')

df[df['description_len'] > 0]['description_len'].clip(upper=2000).hist(
    bins=50, ax=axes[1], color='darkorange', edgecolor='white'
)
axes[1].set_title('Description text length (non-empty, clipped 2k chars)')
axes[1].set_xlabel('characters')

plt.tight_layout()
plt.show()

In [ ]:
# Sample feature bullets from 3 products
for r in [r for r in records if r.get('features')][:3]:
    print(f"Title: {r['title']}")
    for f in r['features']:
        print(f'  • {f}')
    print()

## 6. Price and rating distributions

In [ ]:
import numpy as np

prices        = [r['price'] for r in records if r.get('price') is not None]
ratings       = [r['average_rating'] for r in records if r.get('average_rating') is not None]
rating_counts = [r['rating_number'] for r in records if r.get('rating_number') is not None]

print(f'Price coverage:  {len(prices)/len(records):.1%}  (n={len(prices):,})')
print(f'Rating coverage: {len(ratings)/len(records):.1%}  (n={len(ratings):,})')
print()
if prices:
    arr = np.array(prices)
    print(f'Price   — median: ${np.median(arr):.2f}  mean: ${np.mean(arr):.2f}  max: ${np.max(arr):.2f}')
if ratings:
    arr = np.array(ratings)
    print(f'Rating  — median: {np.median(arr):.2f}  mean: {np.mean(arr):.2f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

if prices:
    pd.Series(prices).clip(upper=500).hist(bins=60, ax=axes[0], color='steelblue', edgecolor='white')
    axes[0].set_title('Price distribution (clipped $500)')
    axes[0].set_xlabel('USD')

if ratings:
    pd.Series(ratings).hist(bins=20, ax=axes[1], color='darkorange', edgecolor='white')
    axes[1].set_title('Average rating')
    axes[1].set_xlabel('stars')

if rating_counts:
    pd.Series(rating_counts).clip(upper=5000).hist(bins=50, ax=axes[2], color='seagreen', edgecolor='white')
    axes[2].set_title('Rating count (clipped 5k)')
    axes[2].set_xlabel('# ratings')

plt.tight_layout()
plt.show()

## 7. `details` field — structured specs

In [ ]:
import json
from collections import Counter

def parse_details(row):
    d = row.get('details')
    if not d:
        return {}
    if isinstance(d, dict):
        return d
    if isinstance(d, str):
        try:
            return json.loads(d)
        except Exception:
            return {}
    return {}

all_detail_keys = Counter()
for row in records:
    all_detail_keys.update(parse_details(row).keys())

print('Top 20 detail keys:')
for key, count in all_detail_keys.most_common(20):
    print(f'  {count:5,}  {key}')

In [ ]:
rich_rows = [r for r in records if len(parse_details(r)) >= 5]
print(f'Products with 5+ detail keys: {len(rich_rows):,}\n')

for r in rich_rows[:3]:
    print('=' * 60)
    print(f"Title:   {r['title']}")
    print(json.dumps(parse_details(r), indent=2)[:600])
    print()

## 8. Draft document — what we'd actually embed

In [ ]:
def build_document(row):
    parts = [row.get('title', '').strip()]

    feats = row.get('features') or []
    if feats:
        parts.append('Features: ' + ' | '.join(feats))

    desc = row.get('description') or []
    if isinstance(desc, list) and desc:
        parts.append('Description: ' + ' '.join(desc))

    details = parse_details(row)
    if details:
        spec_str = ', '.join(f'{k}: {v}' for k, v in list(details.items())[:6])
        parts.append(f'Specs: {spec_str}')

    if row.get('price'):
        parts.append(f"Price: ${row['price']:.2f}")
    if row.get('average_rating'):
        n = row.get('rating_number', 0) or 0
        parts.append(f"Rating: {row['average_rating']}/5 ({n:,} ratings)")

    return '\n'.join(parts)

for r in [r for r in records if features_text(r) or description_text(r)][:5]:
    print('=' * 60)
    print(build_document(r))
    print()

## 9. Document length — fits all-MiniLM-L6-v2's 256-token window?

In [ ]:
doc_lengths   = [len(build_document(r)) for r in records]
doc_series    = pd.Series(doc_lengths)
approx_tokens = doc_series / 4

print('Document character length:')
print(doc_series.describe().to_string())
print()
print('Approx tokens (chars / 4):')
print(approx_tokens.describe().to_string())
print()
print(f'Docs likely exceeding 256-token window: {(approx_tokens > 256).mean():.1%}')
print('  → sentence-transformers truncates silently; that is fine')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
approx_tokens.clip(upper=1000).hist(bins=60, ax=ax, color='steelblue', edgecolor='white')
ax.axvline(256, color='red', linestyle='--', linewidth=1.5, label='256-token limit (all-MiniLM-L6-v2)')
ax.set_xlabel('Approx tokens')
ax.set_title('Document length distribution (clipped at 1k tokens)')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Verdict

| Question | Answer |
|---|---|
| `features` fill rate | ? |
| `description` fill rate | ? |
| Has features OR description | ? |
| Median document length (tokens) | ? |
| % docs over 256-token window | ? |
| Price coverage | ? |
| Rating coverage | ? |
| Good enough for ChatShop Cycle 1? | ✅ / ❌ |